# Telegram Username Checker

1. Edit **Config** cell below
2. Run **Config**, then **Run**
3. On first login — enter the **code from Telegram**

**Modes:**
- `dictionary` — English words as usernames (`@apple`, `@dragon`)
- `random` — random character strings (`@xK9mP`)

Dictionary mode checks **50–100 words per run**. Already checked words are skipped if `SKIP_ALREADY_CHECKED = True`.

## Config

Run this cell first. Change only the values you need.

In [ ]:
# ── Telegram API (https://my.telegram.org) ──────────────────
API_ID = 12345678
API_HASH = "your_api_hash_here"
PHONE = "+1234567890"
PASSWORD_2FA = "your_2fa_password"

# ── Generation mode ─────────────────────────────────────────
GENERATION_MODE = "dictionary"  # "dictionary" or "random"

# dictionary mode
WORDLIST_FILE = "words_alpha.txt"
WORDS_PER_RUN_MIN = 50
WORDS_PER_RUN_MAX = 100
MIN_WORD_LENGTH = 5   # Telegram minimum username length
MAX_WORD_LENGTH = 32  # Telegram maximum username length
SKIP_ALREADY_CHECKED = True  # skip words from checked_words.txt
CHECKED_FILE = "checked_words.txt"  # tracks all previously checked words

# random mode
NICK_COUNT_MIN = 50
NICK_COUNT_MAX = 100
LETTER_COUNT = 5
PREFIX = ""
ALLOW_LOWERCASE = True
ALLOW_UPPERCASE = True
ALLOW_DIGITS = True
FIRST_CHAR_MUST_BE_LETTER = True

# ── Output files ────────────────────────────────────────────
INPUT_FILE = "Input.txt" # the file with usernames
AVAILABLE_FILE = "Available.txt" # the file with available usernames
OVERWRITE_INPUT = True # overwrite the input file?
APPEND_AVAILABLE = False # False = overwrite Available.txt with free usernames from this run only, True  = append free usernames from this run to Available.txt

# ── Checking ────────────────────────────────────────────────
CHECK_DELAY_SEC = 1

# ── Session ─────────────────────────────────────────────────
SESSION_NAME = "telegram"

print("Config loaded.")

## Run

Run this cell after Config.

In [ ]:
import asyncio
import os
import random
import string
import subprocess
import sys
import urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "telethon"])

from telethon import TelegramClient
from telethon.errors import (
    FloodWaitError,
    SessionPasswordNeededError,
    UsernameInvalidError,
    UsernamePurchaseAvailableError,
)
from telethon.tl.functions.account import CheckUsernameRequest

WORDLIST_URL = "https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt"


def load_checked_words() -> set[str]:
    checked = set()
    if os.path.exists(CHECKED_FILE):
        with open(CHECKED_FILE, encoding="utf-8") as f:
            checked.update(line.strip().lstrip("@").lower() for line in f if line.strip())
    return checked


def save_checked_words(names: list[str]) -> None:
    with open(CHECKED_FILE, "a", encoding="utf-8") as f:
        for name in names:
            f.write(name.lstrip("@").lower() + "\n")


def ensure_wordlist() -> None:
    if os.path.exists(WORDLIST_FILE):
        return
    print(f"    Downloading {WORDLIST_FILE}...")
    urllib.request.urlretrieve(WORDLIST_URL, WORDLIST_FILE)


def load_dictionary_words(checked: set[str]) -> list[str]:
    ensure_wordlist()
    words = []
    with open(WORDLIST_FILE, encoding="utf-8") as f:
        for line in f:
            word = line.strip().lower()
            if not word:
                continue
            if not (MIN_WORD_LENGTH <= len(word) <= MAX_WORD_LENGTH):
                continue
            if not word.isalpha():
                continue
            if SKIP_ALREADY_CHECKED and word in checked:
                continue
            words.append(word)
    return words


def pick_dictionary_usernames(checked: set[str]) -> list[str]:
    pool = load_dictionary_words(checked)
    count = random.randint(WORDS_PER_RUN_MIN, WORDS_PER_RUN_MAX)
    if len(pool) < count:
        raise ValueError(
            f"Not enough unchecked words left ({len(pool)}). "
            f"Need {count}. Set SKIP_ALREADY_CHECKED = False or use a fresh CHECKED_FILE."
        )
    return [f"@{word}" for word in random.sample(pool, count)]


def build_charset() -> str:
    charset = ""
    if ALLOW_LOWERCASE:
        charset += string.ascii_lowercase
    if ALLOW_UPPERCASE:
        charset += string.ascii_uppercase
    if ALLOW_DIGITS:
        charset += string.digits
    if not charset:
        raise ValueError("Enable at least one of ALLOW_LOWERCASE, ALLOW_UPPERCASE, ALLOW_DIGITS")
    if FIRST_CHAR_MUST_BE_LETTER and not any(c.isalpha() for c in charset):
        raise ValueError("FIRST_CHAR_MUST_BE_LETTER requires letters in charset")
    return charset


def generate_random_username(charset: str) -> str:
    letters = "".join(c for c in charset if c.isalpha())
    while True:
        random_part = "".join(random.choice(charset) for _ in range(LETTER_COUNT))
        body = f"{PREFIX}{random_part}"
        if FIRST_CHAR_MUST_BE_LETTER and body and not body[0].isalpha():
            body = random.choice(letters) + body[1:]
        if len(body) >= 5:
            return f"@{body}"


def pick_random_usernames() -> list[str]:
    charset = build_charset()
    count = random.randint(NICK_COUNT_MIN, NICK_COUNT_MAX)
    usernames = set()
    while len(usernames) < count:
        usernames.add(generate_random_username(charset))
    return sorted(usernames)


SESSION_FILE = f"{SESSION_NAME}.session"
checked = load_checked_words() if SKIP_ALREADY_CHECKED else set()

print("[1/4] Connecting to Telegram...")
client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
await client.connect()

if not await client.is_user_authorized():
    await client.send_code_request(PHONE)
    code = input("Enter code from Telegram: ")
    try:
        await client.sign_in(PHONE, code)
    except SessionPasswordNeededError:
        await client.sign_in(password=PASSWORD_2FA)

me = await client.get_me()
print(f"OK: {me.first_name} | id={me.id}")

if GENERATION_MODE == "dictionary":
    usernames = pick_dictionary_usernames(checked)
    usernames = sorted(usernames)
    print(f"[2/4] Dictionary mode: picked {len(usernames)} words ({WORDS_PER_RUN_MIN}-{WORDS_PER_RUN_MAX})")
    print(f"    already checked: {len(checked)} | skip enabled: {SKIP_ALREADY_CHECKED}")
elif GENERATION_MODE == "random":
    usernames = pick_random_usernames()
    print(f"[2/4] Random mode: generated {len(usernames)} usernames ({NICK_COUNT_MIN}-{NICK_COUNT_MAX})")
    print(f"    format: @{PREFIX}{'?' * LETTER_COUNT}")
else:
    raise ValueError('GENERATION_MODE must be "dictionary" or "random"')

input_mode = "w" if OVERWRITE_INPUT else "a"
with open(INPUT_FILE, input_mode, encoding="utf-8") as f:
    if input_mode == "a" and os.path.exists(INPUT_FILE) and os.path.getsize(INPUT_FILE) > 0:
        f.write("\n")
    f.write("\n".join(usernames) + "\n")
print("Examples:", ", ".join(usernames[:5]))

print("[3/4] Checking availability...")
available = []
total = len(usernames)
for i, username in enumerate(usernames, 1):
    name = username.lstrip("@")
    try:
        is_free = await client(CheckUsernameRequest(username=name))
        status = "AVAILABLE" if is_free else "taken"
        if is_free:
            available.append(username)
    except UsernameInvalidError:
        status = "invalid"
    except UsernamePurchaseAvailableError:
        status = "auction"
    except FloodWaitError as e:
        await asyncio.sleep(e.seconds)
        is_free = await client(CheckUsernameRequest(username=name))
        status = "AVAILABLE" if is_free else "taken"
        if is_free:
            available.append(username)
    print(f"[{i}/{total}] @{name} — {status}")
    await asyncio.sleep(CHECK_DELAY_SEC)

available_mode = "a" if APPEND_AVAILABLE else "w"
with open(AVAILABLE_FILE, available_mode, encoding="utf-8") as f:
    if available_mode == "a" and os.path.exists(AVAILABLE_FILE) and os.path.getsize(AVAILABLE_FILE) > 0:
        f.write("\n")
    f.write("\n".join(available) + ("\n" if available else ""))

if SKIP_ALREADY_CHECKED:
    save_checked_words(usernames)

await client.disconnect()

print(f"[4/4] Done! Available: {len(available)}")
print(f"  {INPUT_FILE} — all generated usernames")
print(f"  {AVAILABLE_FILE} — available usernames")
if SKIP_ALREADY_CHECKED:
    print(f"  {CHECKED_FILE} — all checked words so far")
print(f"  {SESSION_FILE} — session file (keep for next run)")